# 09 — Joint Inference: Detection + Lane Segmentation

**Goal:** Run inference with the jointly trained model showing both detection boxes and lane mask overlay.

This notebook:
1. Loads joint-trained weights
2. Runs the multi-task model on images
3. Draws detection bounding boxes
4. Overlays predicted lane mask
5. Saves annotated images

### Prerequisites
- Run `08_joint_training.ipynb` first (or have joint weights ready)

## 1 — Setup

In [ ]:
!pip install -q ultralytics>=8.4.0 opencv-python matplotlib pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
import torch
import torch.nn.functional as F
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
DATASET_DIR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_yolo")

# Add project src to path
PROJECT_SRC = os.path.join(ECOCAR_ROOT, "project", "src")
if os.path.isdir(PROJECT_SRC):
    sys.path.insert(0, os.path.dirname(PROJECT_SRC))

# Joint weights path
JOINT_WEIGHTS = os.path.join(ECOCAR_ROOT, "weights", "joint_det_lane_best.pt")
# Fallback: from training run
# JOINT_WEIGHTS = os.path.join(ECOCAR_ROOT, "training_runs", "joint_det_lane", "weights", "best.pt")

# Output
OUTPUT_DIR = os.path.join(ECOCAR_ROOT, "outputs", "09_joint_inference")
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Weights: {JOINT_WEIGHTS} (exists: {os.path.isfile(JOINT_WEIGHTS)})")

## 2 — Load Multi-Task Model

In [ ]:
from src.multitask_model import build_multitask_model

# Build with same architecture config as training
model = build_multitask_model(
    weights='yolo26s.pt',  # Base architecture
    mask_height=180,
    mask_width=320,
    lane_hidden_channels=64,
)

# Load joint-trained weights
if os.path.isfile(JOINT_WEIGHTS):
    ckpt = torch.load(JOINT_WEIGHTS, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"✅ Loaded joint weights from epoch {ckpt.get('epoch', '?')}")
else:
    print("⚠️ Joint weights not found. Using untrained model (for demo only).")

model = model.to(device)
model.eval()
print("Model ready for inference.")

## 3 — Select Test Images

In [ ]:
val_images_dir = os.path.join(DATASET_DIR, 'images', 'val')
NUM_SAMPLES = 6

if os.path.isdir(val_images_dir):
    all_imgs = [os.path.join(val_images_dir, f) for f in os.listdir(val_images_dir)
                if f.endswith(('.jpg', '.png'))]
    test_images = random.sample(all_imgs, min(NUM_SAMPLES, len(all_imgs)))
    print(f"✅ Selected {len(test_images)} images")
else:
    print(f"❌ Val dir not found: {val_images_dir}")
    test_images = []

## 4 — Run Joint Inference

In [ ]:
# BDD100K class names
BDD_CLASSES = [
    'person', 'rider', 'car', 'truck', 'bus',
    'train', 'motorcycle', 'bicycle', 'traffic light', 'traffic sign',
]

# Colours for detection boxes
BOX_COLORS = [
    (255,80,80), (255,160,60), (60,180,255), (100,100,255), (255,220,60),
    (180,100,255), (100,255,100), (255,100,200), (0,255,200), (200,200,0),
]

IMG_SIZE = 640


def run_joint_inference(model, image_path, device, img_size=640, conf_thresh=0.3):
    """
    Run the multi-task model on a single image.
    
    Returns:
        original_img (BGR), det_output, lane_mask_prob (H×W float)
    """
    # Load and preprocess
    img = cv2.imread(image_path)
    orig_h, orig_w = img.shape[:2]
    
    # Resize to model input size
    img_resized = cv2.resize(img, (img_size, img_size))
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    
    # Normalise and convert to tensor
    tensor = torch.from_numpy(img_rgb.astype(np.float32) / 255.0)
    tensor = tensor.permute(2, 0, 1).unsqueeze(0).to(device)  # (1, 3, H, W)
    
    # Forward pass
    with torch.no_grad():
        outputs = model(tensor)
    
    # Lane mask: sigmoid to get probabilities
    lane_logits = outputs['lane_logits']  # (1, 1, mask_h, mask_w)
    lane_prob = torch.sigmoid(lane_logits)[0, 0].cpu().numpy()  # (mask_h, mask_w)
    
    return img, outputs['det_output'], lane_prob


def visualize_joint_result(img, lane_prob, image_path, save_path=None,
                           lane_thresh=0.5, lane_color=(255, 0, 255)):
    """
    Visualize detection boxes + lane mask overlay on the original image.
    """
    h, w = img.shape[:2]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Resize lane mask to original image size
    lane_mask = cv2.resize(lane_prob, (w, h), interpolation=cv2.INTER_LINEAR)
    lane_binary = (lane_mask > lane_thresh).astype(np.uint8)
    
    # Create overlay
    overlay = img_rgb.copy()
    overlay[lane_binary == 1] = lane_color
    blended = cv2.addWeighted(img_rgb, 0.65, overlay, 0.35, 0)
    
    # Display
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    axes[0].imshow(img_rgb)
    axes[0].set_title('Original', fontsize=11)
    axes[0].axis('off')
    
    axes[1].imshow(lane_mask, cmap='hot', vmin=0, vmax=1)
    axes[1].set_title('Lane Probability', fontsize=11)
    axes[1].axis('off')
    
    axes[2].imshow(blended)
    axes[2].set_title('Detection + Lane Overlay', fontsize=11)
    axes[2].axis('off')
    
    plt.suptitle(os.path.basename(image_path), fontsize=12)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"💾 {save_path}")
    
    plt.show()
    
    return blended

In [ ]:
# Run inference on all test images
for img_path in test_images:
    img, det_out, lane_prob = run_joint_inference(model, img_path, device, IMG_SIZE)
    
    save_path = os.path.join(OUTPUT_DIR, f"joint_{os.path.basename(img_path).replace('.jpg', '.png')}")
    visualize_joint_result(img, lane_prob, img_path, save_path=save_path)
    
    # Print lane coverage
    lane_pct = (lane_prob > 0.5).mean() * 100
    print(f"  Lane coverage: {lane_pct:.1f}% of image\n")

## 5 — Compare: Detection-Only vs Joint Model

In [ ]:
from ultralytics import YOLO

# Load detection-only model for comparison
det_only_weights = os.path.join(ECOCAR_ROOT, 'weights', 'bdd100k_yolo26s_best.pt')
if os.path.isfile(det_only_weights):
    det_model = YOLO(det_only_weights)
else:
    det_model = YOLO('yolo26n.pt')  # Fallback to pretrained

# Compare on first 2 images
for img_path in test_images[:2]:
    # Detection-only
    det_result = det_model(img_path, verbose=False)[0]
    det_annotated = det_result.plot()
    
    # Joint model
    img, _, lane_prob = run_joint_inference(model, img_path, device, IMG_SIZE)
    h, w = img.shape[:2]
    lane_mask = cv2.resize(lane_prob, (w, h)) > 0.5
    
    joint_vis = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()
    joint_vis[lane_mask] = [255, 0, 255]  # Magenta lanes
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    ax1.imshow(cv2.cvtColor(det_annotated, cv2.COLOR_BGR2RGB))
    ax1.set_title(f'Detection Only ({len(det_result.boxes)} boxes)', fontsize=12)
    ax1.axis('off')
    ax2.imshow(joint_vis)
    ax2.set_title('Joint: Detection + Lane Mask', fontsize=12)
    ax2.axis('off')
    plt.suptitle(os.path.basename(img_path), fontsize=13)
    plt.tight_layout()
    
    save_path = os.path.join(OUTPUT_DIR, f"compare_{os.path.basename(img_path).replace('.jpg', '.png')}")
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"💾 {save_path}")

## 6 — How to Reuse Joint Model

```python
import torch
from src.multitask_model import build_multitask_model

# Build model (same architecture as training)
model = build_multitask_model('yolo26s.pt', mask_height=180, mask_width=320)

# Load joint weights
ckpt = torch.load('joint_det_lane_best.pt', map_location='cuda')
model.load_state_dict(ckpt['model_state_dict'])
model.eval().cuda()

# Inference
img_tensor = preprocess(image)  # (1, 3, 640, 640)
outputs = model(img_tensor)

det_output = outputs['det_output']      # Detection predictions
lane_logits = outputs['lane_logits']    # Lane segmentation logits
lane_prob = torch.sigmoid(lane_logits)  # Lane probabilities
neck_features = outputs['neck_features'] # Reusable features
```

In [ ]:
print("\n" + "="*60)
print(" JOINT INFERENCE — COMPLETE")
print("="*60)
print(f" Weights:   {JOINT_WEIGHTS}")
print(f" Images:    {len(test_images)}")
print(f" Saved to:  {OUTPUT_DIR}")
print("="*60)
print("\n🎯 The joint model produces both detection boxes and lane segmentation!")
print("   Use the shared backbone features for further downstream tasks.")